# WaferAI — Train EfficientNetB0 on WM-811K

End-to-end training notebook. Produces `wafer_classifier.h5` that you drop into `models/` of your local WaferAI repo for real inference.

**Prerequisites**
1. Upload `LSWMD.pkl` (~2 GB) to the root of your Google Drive.
2. **Runtime → Change runtime type → T4 GPU** (free tier).
3. Run all cells. Total time: ~45–90 min on T4.

The preprocessing here mirrors `scripts/extract_samples.py` so the saved model expects the same input format the local Gradio app sends at inference time.


In [ ]:
import tensorflow as tf
print("TF version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus)
assert gpus, "No GPU detected. Runtime → Change runtime type → T4 GPU."


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_PATH = "/content/drive/MyDrive/LSWMD.pkl"


## 2. Load LSWMD and filter to labelled rows

WM-811K has 811,457 wafer maps; only ~172k carry a defect label. Unlabelled rows are dropped for supervised training.


In [ ]:
import numpy as np
import pandas as pd

print(f"Loading {DATASET_PATH} ...")
df = pd.read_pickle(DATASET_PATH)
print(f"Total rows: {len(df):,}")

def _label(val):
    if isinstance(val, (list, np.ndarray)):
        return val[0] if len(val) else None
    return val

df["label"] = df["failureType"].apply(_label)
df = df[df["label"].notna()].copy()
df["label"] = df["label"].astype(str)

CLASS_NAMES = [
    "Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc",
    "Random", "Scratch", "Near-full", "none",
]
df = df[df["label"].isin(CLASS_NAMES)].reset_index(drop=True)

print(f"Labelled rows: {len(df):,}")
print(df["label"].value_counts())


## 3. Preprocess wafer maps to 96×96×3

WM-811K wafer codes: `0` = outside wafer, `1` = good die, `2` = defect. We render each map to the same RGB palette `scripts/extract_samples.py` uses, so the model trained here expects the exact format the local Gradio app produces.


In [ ]:
from PIL import Image

IMG_SIZE = 96

def wafer_to_rgb(wafer_map):
    h, w = wafer_map.shape
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[wafer_map == 1] = (90, 90, 110)
    img[wafer_map == 2] = (230, 60, 60)
    pil = Image.fromarray(img).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    return np.asarray(pil, dtype=np.uint8)

print("Rendering wafers (~5 min for 172k) ...")
X = np.empty((len(df), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
y_labels = np.empty(len(df), dtype=np.int64)
label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}

for i, row in enumerate(df.itertuples(index=False)):
    X[i] = wafer_to_rgb(row.waferMap)
    y_labels[i] = label_to_idx[row.label]
    if (i + 1) % 20000 == 0:
        print(f"  {i+1:,}/{len(df):,}")

print("X:", X.shape, X.dtype, "  y:", y_labels.shape)


## 4. Stratified 70/15/15 train/val/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y_labels, test_size=0.30, stratify=y_labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)
del X, X_tmp, y_labels, y_tmp

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")


## 5. Build EfficientNetB0 (transfer learning)

ImageNet-pretrained backbone (frozen) → GlobalAveragePooling → Dense(256) → Dropout → Dense(9, softmax). The `Rescaling` layer normalises uint8 inputs inside the model.


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model

def build_model(num_classes=9):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype="uint8")
    x = layers.Rescaling(1.0 / 255.0)(inputs)
    base = EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = False
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs, outputs), base

model, base_model = build_model(len(CLASS_NAMES))
model.summary()


## 6. Train with class weights

WM-811K is extremely imbalanced (~78% are `none`). Class weights normalise the loss so minority classes aren't drowned out.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

class_weights_vals = compute_class_weight(
    "balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=y_train,
)
class_weight = {i: float(w) for i, w in enumerate(class_weights_vals)}
print("Class weights:", class_weight)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=128,
    epochs=10,
    class_weight=class_weight,
    callbacks=callbacks,
)


## 7. Fine-tune (recommended)

Unfreeze the last ~30 layers of the backbone, drop the LR to 1e-5, train 5 more epochs. Typically adds 1–3% absolute accuracy.


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_ft = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=64,
    epochs=5,
    class_weight=class_weight,
    callbacks=callbacks,
)


## 8. Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc*100:.2f}%   Test loss: {test_loss:.4f}\n")

y_pred = model.predict(X_test, batch_size=128, verbose=0).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.show()


## 9. Save and download

The saved file goes into `models/wafer_classifier.h5` of your local WaferAI repo. Set `MODEL_PATH=models/wafer_classifier.h5` in `.env` (already the default) and the app will load it on next launch.


In [ ]:
model.save("wafer_classifier.h5")
print("Saved.")

from google.colab import files
files.download("wafer_classifier.h5")
files.download("confusion_matrix.png")
